# Shakespeare's Characters

The goal of this notebook is to train a simple LSTM model **to mimic the dialogue of Shakespearean characters.** We will train the model on a large corpus of text from Shakespeare's plays, teaching it **to predict the next character in a sequence.** For example, given the input 'to be or no', the model should predict 't' as the most likely following character. The input window then slides forward by one position—now reading 'o be or not'—to predict the next token. This sliding window approach allows the model to generate text character-by-character continuously.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np

In [ ]:
# Setup device
device = None   # TODO

## Data Loading

In [ ]:
# Download and prepare data
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
text = open('input.txt', 'r').read()

In [ ]:
# Create character-level vocabulary
chars = sorted(list(set(text)))

vocab_size = None   # TODO
stoi = None   # TODO  map each char to integer
itos = {i: ch for i, ch in enumerate(chars)}

# Encode the entire text into integers
data = [stoi[c] for c in text]

In [ ]:
# Sliding Window Dataset
# Instead of sentences, we slice the giant text into overlapping sequences

class CharDataset(Dataset):
    def __init__(self, data, seq_len=100):
        self.data = None  # TODO
        self.seq_len = None # TODO

    def __len__(self):
        return None   # TODO remember that the last seq_len tokens are not valid elements, as for them there is no next token to be predicted

    def __getitem__(self, idx):
        # x is a sequence of seq_len characters
        # y is the single character that comes right after it
        x = None # TODO x should be data from idx to idx + seq_len (not included)
        y = None # TODO y should be the single next char to x
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

In [ ]:
# Parameters
seq_len = 100
batch_size = 128

# Create the dataset
dataset = None  # TODO

# Create the data loader
loader = None   # TODO

## Model Definition

In [ ]:
# TODO define your LSTM class

In [ ]:
# Instantiate the model
model = None  # TODO

##Training

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.002)

In [ ]:
epochs = 5
model.train()

for epoch in range(epochs):
    total_loss = 0
    for i, (x, y) in enumerate(loader):
        x, y = None # TODO move x and y to correct device

        # TODO clear out old gradients using optimizer

        logits = None   # TODO compute the output for the model

        loss = None     # TODO compute loss between logits and ground truth

        # TODO compute how much each param contributes to the loss
        # TODO adjust the parameters using the optimizer

        total_loss += loss.item()

        if i % 500 == 0:
            print(f"Epoch {epoch+1}, Batch {i}, Loss: {loss.item():.4f}")

    print(f"--- Epoch {epoch+1} Avg Loss: {total_loss/len(loader):.4f} ---")

## Validation

In [ ]:
def generate_text(model, start_str="ROMEO: ", length=200, temperature=0.8):
    model.eval()
    generated = start_str

    for _ in range(length):
        # Format input string to length seq_len
        input_tokens = [stoi[c] for c in generated[-seq_len:]]
        x = torch.tensor(input_tokens).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(x)

            # Apply temperature and sample (instead of argmax)
            # Higher temp = more "creative"/chaotic text
            probs = torch.softmax(logits / temperature, dim=-1)
            pred_idx = torch.multinomial(probs, num_samples=1).item()

            generated += itos[pred_idx]

    return generated

In [ ]:
print("\nGenerated Shakespeare:\n")
print(generate_text(model, start_str="KING: ", length=300))